In [ ]:
import pathlib
import os
from typing import List
from datasets import load_dataset, concatenate_datasets
import evaluate
import torch
from transformers import (
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from hf_wrapper import GPTForSequenceClassification
from model import GPT, GPTConfig
from tokenizer import load_tokenizer, eod_token
from sklearn.metrics import precision_score, recall_score, f1_score


def load_pretrained_model(path: pathlib.Path, device: str = 'cuda') -> GPT:
    checkpoint = torch.load(path, map_location=device)
    gptconf = GPTConfig(**checkpoint['model_args'])
    model = GPT(gptconf)
    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'
    for k in list(state_dict.keys()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    filtered = {k: v for k, v in state_dict.items() if k in model.state_dict() and v.shape == model.state_dict()[k].shape}
    model.load_state_dict({**model.state_dict(), **filtered})
    return model.to(device)


# ---- Config ----
args = {
    'train_datasets': ['iggy12345/xnli-en-ipa', 'iggy12345/xnli-es-ipa'],
    'eval_datasets': {
        'en': 'iggy12345/xnli-en-ipa',
        'es': 'iggy12345/xnli-es-ipa'
    },
    'epochs': 3,
    'context_size': 1024,
    'learning_rate': 2e-5,
    'batch_size': 16,
    'hf_cache_dir': pathlib.Path('cache'),
    'device': 'cuda'
}

models_and_tokenizers = [
    {"model_type": "ipa",
     "model_path": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_medium_50k/ckpt.pt",
     "tokenizer_paths": ("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-eng-spa-ipa-number-preservation-vocab.json",
                         "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-eng-spa-ipa-number-preservation-merges.txt")},
    {"model_type": "normal",
     "model_path": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_medium_50k/ckpt.pt",
     "tokenizer_paths": ("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-eng-spa-normal-number-preservation-vocab.json",
                         "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-eng-spa-normal-number-preservation-merges.txt")},
]

main_output_dir = pathlib.Path("./training_outputs_both2each")
os.makedirs(main_output_dir, exist_ok=True)

for config in models_and_tokenizers:
    model_type = config["model_type"]
    model_path = config["model_path"]
    vocab_path, merges_path = config["tokenizer_paths"]
    tokenizer = load_tokenizer(vocab_path, merges_path)

    output_dir = main_output_dir / f"output_{model_type}"
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n🔄 Loading {model_type.upper()} model and tokenizer")
    base_model = load_pretrained_model(pathlib.Path(model_path), args['device'])
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.padding_side = tokenizer.padding_side
    model = GPTForSequenceClassification(base_model, num_classes=3).to(args['device'])

    def flatten_multi_features(examples, features: List[str]) -> List[str]:
        separator = f'\n\n{eod_token}\n\n'
        return [separator.join(x) for x in zip(*[examples[f] for f in features])]

    def preprocess_function(examples):
        if model_type == "ipa":
            feature = flatten_multi_features(examples, ['premise-phoneme', 'hypothesis-phoneme'])
        else:
            feature = flatten_multi_features(examples, ['premise', 'hypothesis'])
        return tokenizer(feature, truncation=True, max_length=args['context_size'])

    # ---- Load and preprocess training datasets
    train_datasets = [load_dataset(ds, split="train", cache_dir=str(args['hf_cache_dir'])) for ds in args['train_datasets']]
    full_train = concatenate_datasets(train_datasets)
    train_encoded = full_train.map(preprocess_function, batched=True)

    # ---- Load evaluation datasets
    eval_encoded = {}
    for lang_key, eval_ds in args['eval_datasets'].items():
        raw_eval = load_dataset(eval_ds, split="validation", cache_dir=str(args['hf_cache_dir']))
        eval_encoded[lang_key] = raw_eval.map(preprocess_function, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    metric = evaluate.load("xnli", "en")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = torch.from_numpy(logits).argmax(dim=-1)
        hf_metrics = metric.compute(predictions=predictions, references=labels)
        hf_metrics["precision"] = precision_score(labels, predictions, average="macro")
        hf_metrics["recall"] = recall_score(labels, predictions, average="macro")
        hf_metrics["f1"] = f1_score(labels, predictions, average="macro")
        return hf_metrics

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=1,
        metric_for_best_model="f1",
        load_best_model_at_end=True,
        learning_rate=args['learning_rate'],
        per_device_train_batch_size=args['batch_size'],
        per_device_eval_batch_size=args['batch_size'],
        num_train_epochs=args['epochs'],
        weight_decay=0.01,
        logging_steps=500,
        logging_dir='./logs',
        fp16=True,
        disable_tqdm=False,
        warmup_ratio=0.3,
        save_safetensors=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_encoded,
        eval_dataset=eval_encoded['en'],  # Initial eval just to activate trainer
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print(f"\n🧠 Training {model_type.upper()} model on EN+ES")
    trainer.train(resume_from_checkpoint=False)

    # ---- Evaluate separately on EN and ES
    for lang_key, dataset in eval_encoded.items():
        print(f"\n🔍 Evaluating on {lang_key.upper()} validation set:")
        results = trainer.evaluate(eval_dataset=dataset)
        print(f"✅ Results on {lang_key.upper()}:\n{results}")



🔄 Loading IPA model and tokenizer
number of parameters: 353.24M


Map: 100%|██████████| 2490/2490 [00:00<00:00, 8769.58 examples/s]
/tmp/slurmtmp.1493678/ipykernel_1228377/1310996848.py:133: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



🧠 Training IPA model on EN+ES


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
500,1.107200,1.181353,0.332932,0.349448,0.332932,0.205866
1000,1.043100,1.055562,0.413253,0.452280,0.413253,0.403521
1500,0.996700,1.027941,0.460643,0.502140,0.460643,0.445683
2000,0.983000,1.059338,0.436948,0.530260,0.436948,0.399902
2500,0.970300,0.979081,0.523293,0.532230,0.523293,0.523511
3000,0.957200,1.012259,0.495582,0.540833,0.495582,0.480028
3500,0.925000,0.967597,0.532129,0.544401,0.532129,0.530578
4000,0.931900,0.961325,0.538554,0.562219,0.538554,0.533829
4500,0.927000,0.938696,0.551807,0.582474,0.551807,0.548638
5000,0.931800,0.942698,0.554217,0.578356,0.554217,0.550450
